# Text Processing, Multiprocessing & XML

1. Sequential text processing
2. Multiprocessing with Pool
3. Incremental XML writing


## Sequential Text Processing (Chunks + Carry-over)


In [ ]:
import io

# Pretend this is a huge text file (we use StringIO to simulate it)
big_text = (
    "The quick brown fox jumps over the lazy dog. "
    "Pack my box with five dozen liquor jugs. "
    "How vexingly quick daft zebras jump! "
    "The five boxing wizards jump quickly."
)

def process_chunks(text, chunk_size=20):
    """
    Reads text in chunks and counts complete words.
    Uses carry-over so words are never split.
    """
    stream = io.StringIO(text)   # simulate a file
    carry = ""                   # leftover from the previous chunk
    word_count = 0
    chunk_num = 0

    while True:
        chunk = stream.read(chunk_size)
        if not chunk:            # nothing left to read, we're done
            break

        chunk_num += 1
        # Prepend the carry-over from the previous round
        combined = carry + chunk

        # Find the last space, everything after it is an incomplete word
        last_space = combined.rfind(" ")

        if last_space == -1:
            # No space found at all, the whole thing is one long fragment
            carry = combined
            safe_text = ""
        else:
            safe_text = combined[:last_space]   # complete words
            carry = combined[last_space + 1:]   # leftover fragment

        words = safe_text.split()
        word_count += len(words)

        print(f"Chunk {chunk_num:2d} | safe: {repr(safe_text):<40} | carry: {repr(carry)}")

    # Don't forget the very last carry-over!
    if carry.strip():
        word_count += len(carry.split())
        print(f"Final carry processed: {repr(carry)}")

    print(f"\nTotal words counted: {word_count}")
    print(f"Cross-check (split all at once): {len(big_text.split())}")

process_chunks(big_text, chunk_size=20)


## Part 2 — Multiprocessing with Pool

In [ ]:
import multiprocessing
import time
import os

# A task that takes some time, counts vowels in a sentence
def count_vowels(sentence):
    """Count the number of vowels in a sentence (simulates real work)."""
    time.sleep(0.1)   # simulate slow I/O or computation
    vowels = sum(1 for ch in sentence.lower() if ch in "aeiou")
    pid = os.getpid()  # which process did this work?
    return (sentence[:30], vowels, pid)

sentences = [
    "The quick brown fox jumps over the lazy dog",
    "A wizard's job is to vex chumps quickly in fog",
    "How razorback-jumping frogs can level six piqued gymnasts",
    "Pack my box with five dozen liquor jugs",
    "The five boxing wizards jump quickly",
    "Sphinx of black quartz judge my vow",
]

# --- Sequential (slow) ---
print("=== Sequential ===")
start = time.time()
results_seq = [count_vowels(s) for s in sentences]
elapsed_seq = time.time() - start
print(f"Done in {elapsed_seq:.2f}s")

# --- Parallel with Pool ---
print("\n=== Parallel (Pool) ===")
start = time.time()

# Pool() uses all available CPU cores by default
# You can specify: Pool(processes=4)
with multiprocessing.Pool() as pool:
    results_par = pool.map(count_vowels, sentences)

elapsed_par = time.time() - start
print(f"Done in {elapsed_par:.2f}s")

# Show results
print("\n--- Results ---")
print(f"{'Sentence':<35} {'Vowels':>6}  {'PID':>7}")
print("-" * 55)
for sentence, vowels, pid in results_par:
    print(f"{sentence:<35} {vowels:>6}  {pid:>7}")

print(f"\n Speedup: {elapsed_seq:.2f}s -> {elapsed_par:.2f}s  "
        f"({elapsed_seq/elapsed_par:.1f}x faster)")
print(f"   Using {multiprocessing.cpu_count()} CPU cores")


## Part 3 — Incremental XML Writing

In [ ]:
# Install lxml if not already available
# (it's usually pre-installed in most environments)
!pip install lxml --quiet

In [ ]:
# --- Part 3: Incremental XML Writing ---

import lxml.etree as ET
import sys

# Some data we want to write, imagine this is millions of rows
students = [
    {"id": 1, "name": "Alice",   "grade": "A"},
    {"id": 2, "name": "Bob",     "grade": "B+"},
    {"id": 3, "name": "Charlie", "grade": "A-"},
    {"id": 4, "name": "Diana",   "grade": "B"},
    {"id": 5, "name": "Eve",     "grade": "A+"},
]

output_file = "students.xml"

# --- Incremental write using xmlfile ---
print("Writing XML incrementally...")

with ET.xmlfile(output_file, encoding="utf-8") as xf:
    xf.write_declaration()                     # <?xml version='1.0' encoding='utf-8'?>

    with xf.element("students"):               # <students> root element
        for s in students:

            # Build one <student> element in memory
            elem = ET.Element("student", id=str(s["id"]))
            name_el = ET.SubElement(elem, "name")
            name_el.text = s["name"]
            grade_el = ET.SubElement(elem, "grade")
            grade_el.text = s["grade"]

            # Write it to disk immediately
            xf.write(elem, pretty_print=True)

            # Clear it from memory right away
            elem.clear()

            print(f"  Written & cleared: student id={s['id']}")

print(f"\nDone! File saved as '{output_file}'")

# Read back and pretty-print the result
print("\n--- File contents ---")
with open(output_file, "r", encoding="utf-8") as f:
    print(f.read())

Writing XML incrementally...
  Written & cleared: student id=1
  Written & cleared: student id=2
  Written & cleared: student id=3
  Written & cleared: student id=4
  Written & cleared: student id=5

Done! File saved as 'students.xml'

--- File contents ---
<?xml version='1.0' encoding='utf-8'?>
<students><student id="1">
  <name>Alice</name>
  <grade>A</grade>
</student>
<student id="2">
  <name>Bob</name>
  <grade>B+</grade>
</student>
<student id="3">
  <name>Charlie</name>
  <grade>A-</grade>
</student>
<student id="4">
  <name>Diana</name>
  <grade>B</grade>
</student>
<student id="5">
  <name>Eve</name>
  <grade>A+</grade>
</student>
</students>
